In [5]:
# === Imports ===

from typing import Tuple

import numpy as np
from numba import jit
from matplotlib import pyplot as plt
import matplotlib.style as mplstyle
from scipy.sparse import csr_matrix

mplstyle.use("./docs/pyscopee.mplstyle")

%matplotlib widget

# === Constants ===

DECIMATION_FACTOR = 10
SPLINE_POLY_DEGREE = 3  # or 1
ZOOM_IN_INDICES = slice(5000, 6000)

# === Main ===

# the regularly sampled signal is loaded
data_regular = np.loadtxt("signal_regular.txt", delimiter=",", skiprows=1)

t_values_regular = data_regular[:, 0]
y_values_regular = data_regular[:, 1]

# the irregularly sampled signal is loaded
data_irregular = np.loadtxt("signal_irregular_with_noise.txt", delimiter=",", skiprows=1)

t_values_irregular = data_irregular[:, 0]
y_values_irregular = data_irregular[:, 2]
noise_stddevs_irregular = data_irregular[:, 3]

In [ ]:
freqs = np.fft.rfftfreq(
    len(y_values_regular), d=t_values_regular[1] - t_values_regular[0]
)
signal_fft = np.fft.rfft(y_values_regular)

sinc_signal = np.sinc(2.0 * 20_000.0 * t_values_regular)
sinc_fft = np.fft.rfft(sinc_signal)

plt.close("all")

fig, ax = plt.subplots()

ax.plot(
    freqs, np.abs(signal_fft) / np.abs(signal_fft).max(), label="FFT of regular signal"
)
ax.plot(freqs, np.abs(sinc_fft) / np.abs(sinc_fft).max(), label="FFT of sinc signal")

In [ ]:
test = np.random.rand(50_000)
indices = np.arange(500, 550)

%timeit test[indices]
%timeit test[500:550]

In [ ]:
def generate_kernel_matrix_specs(
    t_values: np.ndarray,
    t_grid: np.ndarray,
    window_size: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Generates the specifications for the generation of the transposed sinc kernel matrix
    ``A.T`` to compute the matrix product ``A.T @ W @ A`` without forming ``A.T``
    explicitly.
    For this, it finds the distances of between all the grid points ``t_grid`` and the
    values in ``t_values`` if they are not more than ``window_size`` apart.
    Besides, it also evaluates the covariance structure between the grid points which
    is basically the sparsity pattern of ``A.T @ W @ A``.

    Parameters
    ----------
    t_values : :class:`numpy.ndarray` of shape ``(n,)``
        The irregularly sampled time values.
    t_grid : :class:`numpy.ndarray` of shape ``(m,)``
        The regularly sampled grid points.
    window_size : :class:`float`
        The size of the window around each grid point to consider.

    Returns
    -------
    nonzero_covariance_indices : :class:`numpy.ndarray` of shape ``(k, 2)``
        The indices of the grid points between which the covariance is non-zero.
        Its ``i``-th row contains the indices of ``t_grid`` that have non-zero
        covariance with ``t_grid[i]``. Redundant covariances that are already given
        by symmetry are not included.
    distances : :class:`numpy.ndarray` of shape ``(l,)``
        The distances of the values in ``t_values`` to the grid points in ``t_grid``
        that are within the window size. It is a compressed version of the matrix
        ``D.T`` from which the transposed sinc kernel matrix ``A.T`` can be computed
        by element-wise kernel evaluation.
        Please refer to the Notes section for details.
    indices : :class:`numpy.ndarray` of shape ``(m, 2)``
        The indices of the first and last value in ``t_values`` that are within the
        window size around each grid point. Its ``i``-th row contain the indices
        of the first and last value in ``t_values`` that are within the window size
        around ``t_grid[i]``.
        Please refer to the Notes section for details.
    indptr: :class:`numpy.ndarray` of shape ``(m,)``
        The index pointers to the ``distances`` and ``indices`` arrays that determine
        how to unpack the distances and indices for each grid point. Its ``i``-th and
        ``i+1``-th element contain the start and stop index of the ``distances``
        associated with ``t_grid[i]``.
        Please refer to the Notes section for details.

    Notes
    -----
    ``distances``, ``indices``, and ``indptr`` are almost equivalent to the ``data``,
    ``indices``, and ``indptr`` arrays in the sparse CSR matrix format. The only
    difference is that ``indices`` is not a 1D-vector full of individual indices, but
    a 2D-Array where each row contains the indices of the first and last value to store
    in the specific row of the matrix to exploit the fact that these indices are always
    consecutive.

    So to unpack the matrix ``D.T``, the following code can be used GIVEN THE DENSE
    MATRIX FITS INTO MEMORY:

    ```python
    matrix_D = np.zeros(shape=(len(t_grid), len(t_values)))
    for grid_index, (index_from, index_to) in enumerate(indices):
        data_index_from = indptr[grid_index]
        data_index_to = indptr[grid_index + 1]
        matrix_D[grid_index, index_from:index_to] = distances[data_index_from:data_index_to]
    ```

    """  # noqa: E501

    # the range around each of the values in ``t_grid`` is calculated
    # NOTE: they need to be clipped to the minimum and maximum grid values
    t_grid_min, t_grid_max = t_grid[0], t_grid[-1]
    t_grid_range_lower = np.maximum(t_grid - window_size, t_grid_min)
    t_grid_range_upper = np.minimum(t_grid + window_size, t_grid_max)

    # afterwards, the covariance structure between the grid points is calculated, i.e.,
    # the indices at which the matrix ``A.T @ W @ A`` will have non-zero entries
    indices_from = np.searchsorted(t_grid, t_grid_range_lower, side="left")
    # to avoid for double counting, the lower indices are limited to the indices of the
    # respective grid points to avoid redundant entries, e.g., if point number 5 is in
    # the range of point number 2, the covariance between point 5 and point 2 does not
    # need to be computed as it is already given by the covariance between point 2 and
    # point 5
    indices_from = np.maximum(indices_from, np.arange(0, t_grid.size, 1, dtype=indices_from.dtype))
    indices_to = np.searchsorted(t_grid, t_grid_range_upper, side="right")
    nonzero_covariance_indices = np.concatenate(
        (
            indices_from.reshape((-1, 1)),
            indices_to.reshape((-1, 1)),
        ),
        axis=1,
    )

    # now, all the ``t_values``that are within the range of a grid point are found
    half_window_size = window_size / 2.0
    t_grid_range_lower = np.maximum(t_grid - half_window_size, t_grid_min)
    t_grid_range_upper = np.minimum(t_grid + half_window_size, t_grid_max)
    indices_from = np.searchsorted(t_values, t_grid_range_lower, side="left")
    indices_to = np.searchsorted(t_values, t_grid_range_upper, side="right")
    indptr = np.concatenate(
        (
            np.array([0], dtype=indices_from.dtype),
            np.cumsum(indices_to - indices_from)
        ),
    )

    # the distances to the grid points are calculated
    distances = np.empty(shape=(indptr[-1]), dtype=t_values.dtype)
    write_index_to = 0

    for grid_index, grid_value in enumerate(t_grid):
        read_index_from = indices_from[grid_index]
        read_index_to = indices_to[grid_index]
        write_index_from = write_index_to
        write_index_to = indptr[grid_index + 1]

        distances[write_index_from:write_index_to] = t_values[read_index_from:read_index_to] - grid_value

    return (
        nonzero_covariance_indices,
        distances,
        np.concatenate(
            (
                indices_from.reshape((-1, 1)),
                indices_to.reshape((-1, 1)),
            ),
            axis=1,
        ),
        indptr,
    )

@jit(nopython=True)
def sinh_zeromapped_window(x: np.ndarray, exponent: float) -> np.ndarray:
    return np.power(
         np.sinh(1.0 - np.square(x)) / np.sinh(1.0),
         exponent,
     )
    one_minus_x_squared = 1.0 - np.square(x)
    return np.power(
        one_minus_x_squared - (1.0/6.0) * one_minus_x_squared * one_minus_x_squared * one_minus_x_squared,
        exponent,
    )

def promote_distance_matrix_to_sinc_kernel_matrix(
    distances: np.ndarray,
    bandlimit_frequency: float,
    exponent: float,
    window_size: float,
) -> np.ndarray:
    """
    Promotes the transposed distance matrix ``D.T`` to a sinc kernel matrix ``A.T`` by
    computing the sinc kernel values corresponding to the distances and multiplying them
    with the window function values.

    The function ``generate_kernel_matrix_specs`` already limited the distances to
    compute only those that are within the window size around the grid points, so the
    sinc and the window function do not need to be zeroed out here and only nonzero
    entries will be computed.

    """

    return np.sinc(2.0 * bandlimit_frequency * distances) * sinh_zeromapped_window(
        x=(0.5 / window_size) * distances, exponent=exponent,)

def apply_weights_to_sinc_kernel_matrix(
    sinc_kernel_values: np.ndarray,
    t_value_indices: np.ndarray,
    t_value_indptr: np.ndarray,
    weights: np.ndarray,
) -> np.ndarray:
    """
    Appplies the square root of the weights to the sinc kernel values to pre-compute
    the weighted kernel matrix ``A.T @ W``.

    For the arrangement of ``sinc_kernel_values``, ``t_value_indices``, and
    ``t_value_indptr``, please refer to the documentation of the function
    :func:`generate_kernel_matrix_specs`.

    Parameters
    ----------
    sinc_kernel_values : :class:`numpy.ndarray` of shape ``(l,)``
        The values of the sinc kernel function evaluated at the distances between the
        grid points and the values in ``t_values``.
    t_value_indices : :class:`numpy.ndarray` of shape ``(m, 2)``
        The indices of the first and last value in ``t_values`` that are within the
        window size around each grid point.
    t_value_indptr : :class:`numpy.ndarray` of shape ``(m,)``
        The index pointers to the ``sinc_kernel_values`` array that determine how to
        unpack the values for each grid point.
    weights : :class:`numpy.ndarray` of shape ``(n,)``
        The weights to apply to the values in ``t_values``. Its ``j``-th element
        contains the square root of the weight to apply to ``t_values[j]``.

    Returns
    -------
    weighted_sinc_kernel_matrix : :class:`numpy.ndarray` of shape ``(l,)``
        The weighted sinc kernel matrix ``A.T @ W`` that is still stored in the
        same compressed format as ``sinc_kernel_values``.

    """ # noqa: E501

    weighted_sinc_kernel_matrix = np.empty_like(sinc_kernel_values)

    for grid_index, (index_from, index_to) in enumerate(t_value_indices):
        data_index_from = t_value_indptr[grid_index]
        data_index_to = t_value_indptr[grid_index + 1]

        weighted_sinc_kernel_matrix[data_index_from:data_index_to] = (
            sinc_kernel_values[data_index_from:data_index_to] * weights[index_from:index_to]
        )

    return weighted_sinc_kernel_matrix

def matvec_weighted_sinc_kernel_matrix(
    weighted_sinc_kernel_matrix: np.ndarray,
    t_value_indices: np.ndarray,
    t_value_indptr: np.ndarray,
    vector: np.ndarray,
) -> np.ndarray:
    """
    Computes the matrix-vector product of the weighted sinc kernel matrix ``A.T @ W``
    with a vector ``v``, i.e., ``A.T @ W @ v``.

    For the arrangement of ``weighted_sinc_kernel_matrix``, ``t_value_indices``, and
    ``t_value_indptr``, please refer to the documentation of the function
    :func:`apply_weights_to_sinc_kernel_matrix`.

    Parameters
    ----------
    weighted_sinc_kernel_matrix : :class:`numpy.ndarray` of shape ``(l,)``
        The weighted sinc kernel matrix ``A.T @ W`` that is still stored in the
        same compressed format as ``sinc_kernel_values``.
    t_value_indices : :class:`numpy.ndarray` of shape ``(m, 2)``
        The indices of the first and last value in ``t_values`` that are within the
        window size around each grid point.
    t_value_indptr : :class:`numpy.ndarray` of shape ``(m,)``
        The index pointers to the ``sinc_kernel_values`` array that determine how to
        unpack the values for each grid point.
    vector : :class:`numpy.ndarray` of shape ``(n,)``
        The vector ``v`` to multiply with the weighted sinc kernel matrix.

    Returns
    -------
    result : :class:`numpy.ndarray` of shape ``(m,)``
        The result of the matrix-vector product ``A.T @ W @ v``.

    """ # noqa: E501

    result = np.empty(shape=(t_value_indices.shape[0]), dtype=vector.dtype)

    for grid_index, (index_from, index_to) in enumerate(t_value_indices):
        data_index_from = t_value_indptr[grid_index]
        data_index_to = t_value_indptr[grid_index + 1]

        result[grid_index] = (
            weighted_sinc_kernel_matrix[data_index_from:data_index_to] * vector[index_from:index_to]
        ).sum()

    return result

@jit(nopython=True)
def find_index_overlap(
    index_min1: int,
    index_max1: int,
    index_min2: int,
    index_max2: int,
) -> Tuple[int, int, int, int]:
    """
    Finds the overlap between two index ranges.

    Parameters
    ----------
    index_min1, index_max1 : :class:`int`
        The minimum and maximum index of the first range.
    index_min2, index_max2 : :class:`int`
        The minimum and maximum index of the second range.

    Returns
    -------
    index_min, index_max : (:class:`int`, :class:`int`)
        The minimum and maximum index of the overlap between the two ranges.

    """

    index_min = max(index_min1, index_min2)
    index_max = min(index_max1, index_max2)

    return index_min - index_min1, index_max - index_min1, index_min - index_min2, index_max - index_min2

def weighted_sinc_kernel_gram_matrix_as_lapack_banded(
    weighted_sinc_kernel_matrix: np.ndarray,
    sinc_kernel_matrix: np.ndarray,
    t_value_indices: np.ndarray,
    t_value_indptr: np.ndarray,
    covariance_indices: np.ndarray,
) -> np.ndarray:
    """
    Computes the Gram matrix of the weighted sinc kernel matrix ``A.T @ W @ A`` in a
    banded format that can be used with LAPACK's banded solvers.
    Since the matrix is symmetric, only the upper triangular part is computed.

    For the arrangement of ``weighted_sinc_kernel_matrix``, ``sinc_kernel_matrix``,
    ``t_value_indices``, and ``t_value_indptr``, please refer to the documentation of
    the function :func:`generate_kernel_matrix_specs`.

    Parameters
    ----------
    weighted_sinc_kernel_matrix : :class:`numpy.ndarray` of shape ``(l,)``
        The weighted sinc kernel matrix ``A.T @ W`` that is still stored in the
        same compressed format as ``sinc_kernel_values``.
    sinc_kernel_matrix : :class:`numpy.ndarray` of shape ``(l,)``
        The sinc kernel matrix ``A.T`` that is still stored in the same compressed
        format as ``sinc_kernel_values``.
    t_value_indices : :class:`numpy.ndarray` of shape ``(m, 2)``
        The indices of the first and last value in ``t_values`` that are within the
        window size around each grid point.
    t_value_indptr : :class:`numpy.ndarray` of shape ``(m,)``
        The index pointers to the ``sinc_kernel_values`` array that determine how to
        unpack the values for each grid point.
    covariance_indices : :class:`numpy.ndarray` of shape ``(k, 2)``
        The indices of the grid points between which the covariance is non-zero.
        Its ``i``-th row contains the indices of ``t_grid`` that have non-zero
        covariance with ``t_grid[i]``.

    Returns
    -------
    gram_matrix : :class:`numpy.ndarray` of shape ``(bandwidth, m)``
        The Gram matrix of the weighted sinc kernel matrix ``A.T @ W @ A`` in a banded
        format that can be used with LAPACK's banded solvers.

    """

    # the semibandwidth of the banded result is determined
    semibandwidth = (covariance_indices[:, 1] - covariance_indices[:, 0]).max() - 1

    # the Gram matrix is filled in the banded format
    gram_matrix = np.zeros(shape=(semibandwidth + 1, t_value_indices.shape[0]), dtype=weighted_sinc_kernel_matrix.dtype)

    for grid_index, (atw_index_from, atw_index_to) in enumerate(t_value_indices):
        atw_data_index_from = t_value_indptr[grid_index]
        atw_data_index_to = t_value_indptr[grid_index + 1]
        atw_data = weighted_sinc_kernel_matrix[atw_data_index_from:atw_data_index_to]

        cov_index_from, cov_index_to = covariance_indices[grid_index]
        for cov_index in range(cov_index_from, cov_index_to):
            cov_data_index_from = t_value_indptr[cov_index]
            cov_data_index_to = t_value_indptr[cov_index + 1]
            a_index_from, a_index_to = t_value_indices[cov_index]
            cov_data = sinc_kernel_matrix[cov_data_index_from:cov_data_index_to]

            atw_index_min, atw_index_max, a_index_min, a_index_max = find_index_overlap(
                atw_index_from, atw_index_to, a_index_from, a_index_to
            )

            row_index = semibandwidth + (grid_index - cov_index)
            col_index = cov_index

            gram_matrix[row_index, col_index] = (
                atw_data[atw_index_min:atw_index_max] * cov_data[a_index_min:a_index_max]
            ).sum()

    return gram_matrix



generate_kernel_matrix_specs_jit = jit(generate_kernel_matrix_specs, nopython=True)
promote_distance_matrix_to_sinc_kernel_matrix_jit = jit(promote_distance_matrix_to_sinc_kernel_matrix, nopython=True)
apply_weights_to_sinc_kernel_matrix_jit = jit(apply_weights_to_sinc_kernel_matrix, nopython=True)
matvec_weighted_sinc_kernel_matrix_jit = jit(matvec_weighted_sinc_kernel_matrix, nopython=True)
weighted_sinc_kernel_gram_matrix_as_lapack_banded_jit = jit(weighted_sinc_kernel_gram_matrix_as_lapack_banded, nopython=True)

a = np.linspace(0, 10, 50_000)
b = np.linspace(0, 10, 5_000)
print("delta b", b[1] - b[0])

cov_indices, dist, indices, indptr = generate_kernel_matrix_specs_jit(a, b, 5 * (b[1] - b[0]))
new_dist = promote_distance_matrix_to_sinc_kernel_matrix_jit(dist, 2_000, 5.0, 5 * (b[1] - b[0]))

indices_for_csr = np.concatenate(
    tuple(
        np.arange(start, stop)
        for start, stop in indices
    )
)
a_csr = csr_matrix(
    (new_dist, indices_for_csr, indptr),
    shape=(len(b), len(a)),
)

ata = (a_csr @ a_csr.T)

weighted_dist = apply_weights_to_sinc_kernel_matrix_jit(new_dist, indices, indptr, np.ones_like(a))
result = matvec_weighted_sinc_kernel_matrix_jit(weighted_dist, indices, indptr, np.ones_like(a))
gram = weighted_sinc_kernel_gram_matrix_as_lapack_banded(
    weighted_sinc_kernel_matrix=weighted_dist,
    sinc_kernel_matrix=new_dist,
    t_value_indices=indices,
    t_value_indptr=indptr,
    covariance_indices=cov_indices,
)

assert(np.allclose(ata.diagonal(0), gram[gram.shape[0] - 1, :]))
assert(np.allclose(ata.diagonal(1), gram[gram.shape[0] - 2, 1:]))
assert(np.allclose(ata.diagonal(2), gram[gram.shape[0] - 3, 2:]))
assert(np.allclose(ata.diagonal(3), gram[gram.shape[0] - 4, 3:]))

print(ata)
print(gram)

wghts = np.ones_like(a)
%timeit gram = weighted_sinc_kernel_gram_matrix_as_lapack_banded_jit(weighted_dist, new_dist, indices, indptr, cov_indices)
%timeit cov_indices, dist, indices, indptr = generate_kernel_matrix_specs(a, b, 5 * (b[1] - b[0]))
%timeit cov_indices, dist, indices, indptr = generate_kernel_matrix_specs_jit(a, b, 5 * (b[1] - b[0]))
%timeit new_dist = promote_distance_matrix_to_sinc_kernel_matrix(dist, 2_000, 5.0, 5 * (b[1] - b[0]))
%timeit new_dist = promote_distance_matrix_to_sinc_kernel_matrix_jit(dist, 2_000, 5.0, 5 * (b[1] - b[0]))
%timeit weighted_dist = apply_weights_to_sinc_kernel_matrix(new_dist, indices, indptr, wghts)
%timeit weighted_dist = apply_weights_to_sinc_kernel_matrix_jit(new_dist, indices, indptr, wghts)
%timeit result = matvec_weighted_sinc_kernel_matrix(weighted_dist, indices, indptr, wghts)
%timeit result = matvec_weighted_sinc_kernel_matrix_jit(weighted_dist, indices, indptr, wghts)


""" matrix_D = np.zeros(shape=(len(b), len(a)))
for grid_index, (index_from, index_to) in enumerate(indices):
    matrix_D[grid_index, index_from:index_to] = dist[indptr[grid_index]:indptr[grid_index + 1]]

fig, ax = plt.subplots()

ax.imshow(matrix_D, aspect="auto", cmap="viridis") """


print(generate_kernel_matrix_specs(a, b, 0.1))
# print(find_grid_distances_in_window_reach_jit(a, b, 0.1))

# %timeit find_grid_distances_in_window_reach(a, b, 0.1)
# %timeit find_grid_distances_in_window_reach_jit(a, b, 0.1)

In [ ]:
from numba import jit


""" def sinh_zeromapped_window(x: np.ndarray, exponent: float) -> np.ndarray:
    return np.power(
        np.sinh(1.0 - np.square(x)) / np.sinh(1.0),
        exponent,
    ) """


def windowed_sinc_kernel(
    x: np.ndarray, bandlimit_freq: float, window_width: float, window_exponent: float
) -> np.ndarray:
    return np.sinc(
        2.0 * bandlimit_freq * x
    )  # * sinh_zeromapped_window(x, window_width, window_exponent)


plt.close("all")

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()

bandlimit_freq = 20_000.0
exponent = 5.0

# y_values = sinh_zeromapped_window(t_values_regular exponent)
y_values2 = windowed_sinc_kernel(t_values_regular, bandlimit_freq, 0.05, exponent)

freqs = np.fft.rfftfreq(len(y_values), d=t_values_regular[1] - t_values_regular[0])
sinc_window_fft = np.fft.rfft(y_values2)

# ax.plot(t_values_regular, y_values)
ax.plot(t_values_regular, y_values2)

ax2.plot(freqs, np.abs(sinc_window_fft) / np.abs(sinc_window_fft).max())

plt.show()